In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import statsmodels.api as sm

from ISLP.models import (ModelSpec as MS, summarize , poly)
from ISLP import confusion_table

# Classifiers

## Load Data

In [ ]:
default = pd.read_csv("../data/Default.csv")
default['student'] = default['student'].astype('category')
default['default'] = default['default'].astype('category')
default.head()

In [ ]:
fig, ax = plt.subplots()
default.plot.scatter(x='balance', y='income', ax=ax, c='default', colormap='viridis')
ax.set_title('Default by Balance and Income')
plt.show()

In [ ]:
fig, ax = plt.subplots()
default.plot.scatter(x='balance', y='income', ax=ax, c='student', colormap='viridis')
ax.set_title('student by Balance and Income')
plt.show()

## Logistic Classifier

### Single Feature

$$
p(X) = \phi(\beta_0 + \beta_1 X_1)
$$
and try $X_1 = X_{\rm balance}$

In [ ]:
design = MS(['balance'])
X= design.fit_transform(default)
X.head()

For training, format the response $y$ values as booleans:

In [ ]:
y = default['default'] == 'Yes'
y.head()

In [ ]:
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()
summarize(results)

In [ ]:
results.summary()

New predictions: same as with regression

In [ ]:
new_def = pd.DataFrame({'balance': [0, 1000, 2000, 3000, 4000, 5000]})
new_X = design.transform(new_def)
predicted_prob = results.predict(new_X)
predicted_prob

Check error rate of training data

In [ ]:
p_pred = results.predict() # gives the predicted probabilities for the training data
p_pred

Turn these into $\hat{y}_i$, the predicted labels:

In [ ]:
labels_pred = np.full(len(p_pred), 'No', dtype=object)
labels_pred

In [ ]:
pthreshold = 0.5
# switch rows to 'Yes' where the predicted probability is above the threshold
labels_pred[p_pred >= pthreshold] = 'Yes'
labels_pred

In [ ]:
np.mean(labels_pred != default['default']) # accuracy

Error rate is 2.75%

Confusion table/matrix:

In [ ]:
confusion_table(labels_pred, default['default'])

We missed 233/333% who would default, but we did not pick up.  These are false negatives. We have 42 false positives (people who did not fault default, but we would have predicted they would).

### Multiple Features

In [ ]:
design = MS(['balance', 'income','student'])
X = design.fit_transform(default)
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()
results.summary()

In [ ]:
p_pred = results.predict() # gives the predicted probabilities for the training data
labels_pred = np.full(len(p_pred), 'No', dtype=object)
pthreshold = 0.5
# switch rows to 'Yes' where the predicted probability is above the threshold
labels_pred[p_pred >= pthreshold] = 'Yes'
np.mean(labels_pred != default['default']) # accuracy

2.68% instead of 2.75% for the error rate
